In [1]:
import pandas as pd
import duckdb

#from juptils import pretty
#from juptils import lefty

import ipywidgets as widgets
from ipywidgets import interact

## Load monthly sales rollup data.

In [2]:
monthly_sales = pd.read_csv("./csv/monthly-sales.csv")

# Supplier queries

In [4]:
def my_query(supplier):

    supplier_clause = {
        "nippon-metal":"supplier = 'nippon-metal'",
        "us-steel":"supplier = 'us-steel'",
    }.get(supplier, "1=1")

    sql_template = """SELECT 
            SUBSTR(month, 1, 4) AS year, 
            supplier AS supplier,
            format('{:t,}', SUM(month_amt)) AS ytd_amt
        FROM monthly_sales
        WHERE _SUPPLIER_
        GROUP BY year, supplier
        ORDER BY year, supplier
        """.replace("_SUPPLIER_", supplier_clause)
    
    df = duckdb.query(sql_template).df() 
    
    df2 = (
        df.style
        .set_properties(subset=['supplier'], **{'text-align': 'left'})
        .set_properties(subset=['ytd_amt'], **{'text-align': 'right'})
        .hide(axis="index")
    )

    display(df2)

   
interact(my_query, supplier=['all','nippon-metal', 'us-steel']);

interactive(children=(Dropdown(description='supplier', options=('all', 'nippon-metal', 'us-steel'), value='all…

## Yearly Sales

In [5]:
# NOTE: '{:t,}'  works, but '{:t}' doesn't work. for floats: '{:t,.2f}'

df = duckdb.query("""
SELECT 
    SUBSTR(month, 1, 4) AS year, 
    supplier AS supplier, 
    format('{:t,}', SUM(month_amt)) AS ytd_amt

FROM monthly_sales
GROUP BY year, supplier
ORDER BY year, supplier
""").df()

df.style.set_properties(subset=['supplier'],  **{'text-align': 'left'}).set_properties(subset=['ytd_amt'], **{'text-align': 'right'})
    


#.style.set_properties(**{'white-space': 'pre'})
#.lefty() 

#interact(f, x=[('one', 10), ('two', 20)]);

,year,supplier,ytd_amt
0,2023,nippon-metal,"9,360,700"
1,2023,us-steel,"9,675,200"
2,2024,nippon-metal,"9,489,700"
3,2024,us-steel,"8,797,300"
4,2025,nippon-metal,"11,298,600"
5,2025,us-steel,"13,043,700"
6,2026,nippon-metal,"1,153,000"
7,2026,us-steel,"306,300"


# Quarterly Sales

In [3]:
# retro fit quarterly supplier

WHERE_CONDITION_SLUG = "_WHERE_CONDITION_SLUG_"

base_query = """SELECT 
        SUBSTR(month, 1, 4) || '-Q' || CAST(CEIL(CAST(SUBSTR(month, 6, 2) AS INT64) / 3.0) AS INT64) AS quarter,
        supplier AS supplier,
        format('{:t,}', SUM(month_amt)) AS quarter_amt
    FROM monthly_sales
    WHERE _WHERE_CONDITION_SLUG_ 
    GROUP BY quarter, supplier
    ORDER BY quarter, supplier
    """


def where_condition(supplier):
    return  {
        "nippon-metal":"supplier = 'nippon-metal'",
        "us-steel":    "supplier = 'us-steel'",
    }.get(supplier, "1=1")
       

def query_builder(template, slug, replacement):
    return template.replace(slug, replacement)


def query(supplier):    
    
    where_cond = where_condition(supplier)
    
    sql = query_builder(base_query, WHERE_CONDITION_SLUG, where_cond)    

    df = duckdb.query(sql).df() 
    df2 = (
        df.style
        .set_properties(subset=['supplier'], **{'text-align': 'left'})
        .set_properties(subset=['quarter_amt'], **{'text-align': 'right'})
        .hide(axis="index")
    )
    display(df2)
    

interact(query, supplier=['all','nippon-metal', 'us-steel']);

interactive(children=(Dropdown(description='supplier', options=('all', 'nippon-metal', 'us-steel'), value='all…